# Demand Predictor — Train on Colab (GPU)

Trains the `DemandPredictor` U-Net (Stage 3 of the completing-router overhaul). It learns to
predict, per layer, **where the next K nets will route**, so the current net can be routed to leave
space for them. Supervised — labels come from boards completed by the rip-up-and-reroute router.

**Before you run:** make sure these files are pushed to GitHub (they are new):
`pcb_router/routing/rip_up_router.py`, `pcb_router/models/demand_predictor.py`,
`scripts/train_demand_predictor.py`.

**Runtime:** set Runtime → Change runtime type → GPU.

## 0. Setup

In [ ]:
import os, sys
REPO = '/content/Router'
if not os.path.exists(REPO):
    !git clone https://github.com/Klutzhehe/Router.git $REPO
os.chdir(REPO)
!git pull --ff-only
if REPO not in sys.path:
    sys.path.insert(0, REPO)   # chdir alone does NOT put the repo on sys.path; imports need this
!pip -q install scipy pyyaml
import torch
assert os.path.exists('scripts/train_demand_predictor.py'), 'new files missing - did git pull fetch them?'
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), '| files OK')

## 1. Generate the dataset

Routes boards with the rip-up-and-reroute router and keeps only fully-routed boards (clean labels).
Use **multi-layer** stages — they complete far more reliably than single-layer congested ones
(single-layer boards can have pins sealed by other nets' pads). Start small to gauge speed, then
raise `boards_per_stage`.

In [ ]:
from scripts.train_demand_predictor import generate_dataset

samples = generate_dataset(
    stage_names=['s10_via_plus_multi_net'],   # multi-layer; add e.g. 's11_...' for variety
    boards_per_stage=60,
    out_path='data/demand_dataset.pkl',
    max_iters=8,
)
print('boards generated:', len(samples))

## 2. Train the predictor

Weighted BCE (route cells are ~0.1% of pixels, so positives are up-weighted). Batches pad to the
batch's max size, so no wasted compute on small boards. Saves to `checkpoints/demand_predictor.pt`.

In [ ]:
from scripts.train_demand_predictor import train

model = train(
    dataset_path='data/demand_dataset.pkl',
    epochs=80,
    batch_size=8,
    lr=2e-3,
    K=3,
    device='cuda',
    ckpt='checkpoints/demand_predictor.pt',
)

## 3. Sanity check — predicted demand vs. actual future routes

On a trained model the **PREDICTED demand** should light up where the **ACTUAL future routes** are
(and along the corridor between the next-K pins).

In [ ]:
import pickle
import torch
import matplotlib.pyplot as plt
from scripts.train_demand_predictor import DemandDataset, collate_pad
from pcb_router.models.demand_predictor import DemandPredictor

samples = pickle.load(open('data/demand_dataset.pkl', 'rb'))
ds = DemandDataset(samples, K=3)

model = DemandPredictor().cuda()
model.load_state_dict(torch.load('checkpoints/demand_predictor.pt')['model'])
model.eval()

i = 0                        # try different sample indices
x, y, lm = ds[i]
xb, yb, lmb, smb = collate_pad([ds[i]])
with torch.no_grad():
    p = model.predict(xb.cuda(), lmb.cuda())[0].cpu()

L0 = 0
fig, ax = plt.subplots(1, 4, figsize=(20, 5))
ax[0].imshow(x[8]);   ax[0].set_title('current net pins')
ax[1].imshow(x[9]);   ax[1].set_title('next-K nets pins')
ax[2].imshow(p[L0]);  ax[2].set_title('PREDICTED demand (L0)')
ax[3].imshow(y[L0]);  ax[3].set_title('ACTUAL future routes (L0)')
for a in ax:
    a.axis('off')
plt.tight_layout()
plt.show()